# MCP Handbook

Background for the hackathon. [START_HERE.ipynb](START_HERE.ipynb) stays on the lab path: define a tool, call it with servers from the registry, then build your own and open a pull request. This notebook is the longer explanation of the protocol and of the loop the agent uses when it calls your tools.

The registry you are contributing to is the [EVE MCP tool registry](https://github.com/eve-esa/mcp-tool-registry).

**Read this against**

- [MCP architecture](https://modelcontextprotocol.io/docs/learn/architecture) (official overview)
- [MCP specification, 2026-07-28](https://modelcontextprotocol.io/specification/2026-07-28)
- [FastMCP tools](https://gofastmcp.com/servers/tools), [resources](https://gofastmcp.com/servers/resources), and [prompts](https://gofastmcp.com/servers/prompts)
- Yao et al., *ReAct: Synergizing Reasoning and Acting in Language Models*, ICLR 2023. [arXiv:2210.03629](https://arxiv.org/abs/2210.03629)


## Terms

- **Agent**: a language model that decides which tools to call, reads what came back, and repeats until it can answer. In this lab that loop is ReAct, in section 7 below.
- **MCP host**: the application the person is talking to. Today that is EVE. The same server also works in Claude, Cursor, and any other host that speaks MCP.
- **MCP client**: the piece inside the host that holds one connection to one server. A host that uses three servers has three clients.
- **MCP server**: a program that offers tools (and, optionally, resources and prompts). `geocode_mcp` in the lab notebook is one. `"effis"` and `"serpapi"` are others, already published in the registry.
- **Registry**: the [EVE MCP tool registry](https://github.com/eve-esa/mcp-tool-registry). You refer to a published server by its folder name (`"effis"`). The purpose of the hackathon is a pull request that adds or improves a server there.


## 1. What MCP is

**MCP (Model Context Protocol)** is an open protocol for connecting an AI application to external data and tools. The usual picture is **"USB-C for AI"**: one interface, instead of a custom integration for every service and every assistant.

It standardises the messages between the application and the programs that hold data or actions. The host chooses the model and runs the agent loop. Those messages are [JSON-RPC 2.0](https://www.jsonrpc.org/specification). SDKs such as [FastMCP](https://gofastmcp.com) hide the JSON. You write a Python function. The SDK advertises it, checks the arguments, and returns the result.

Three participants, from the [architecture overview](https://modelcontextprotocol.io/docs/learn/architecture):

| Participant | Role | In this hackathon |
|---|---|---|
| **Host** | The AI application. It creates one client per server. | EVE. Also Claude or Cursor, if you point them at the same server. |
| **Client** | Maintains a single connection and fetches context for the host. | Created for you inside `create_agent`. |
| **Server** | A focused program that exposes tools, resources, and prompts. | The server you write, plus registry servers such as `effis`. |

A local server started on your machine typically serves one client, over stdin/stdout. A remote server, reached over HTTP, can serve many clients. The registry deploys servers the second way. Either way the messages are the same, so a tool you test in this notebook is the tool a reviewer runs from the registry.

<img src='./images/mcp-components.png' alt='MCP main components' style='max-width:800px' />

*Fig. 1. MCP main components. Source: [Hugging Face MCP course](https://huggingface.co/learn/mcp-course/unit1/key-concepts).*


## 2. Tools, resources, and prompts

A server can offer three primitives. The specification lists them as [tools](https://modelcontextprotocol.io/specification/2026-07-28/server/tools), [resources](https://modelcontextprotocol.io/specification/2026-07-28/server/resources), and [prompts](https://modelcontextprotocol.io/specification/2026-07-28/server/prompts). The client discovers each list at run time (`tools/list`, `resources/list`, `prompts/list`). Add one, and every connected host can see it. You do not ship a new client.

Who invokes them is different.

### Tools: the model calls them

A tool is a function the model may run to act or to fetch something. Query a STAC catalog, ask Nominatim for a bounding box, read a burnt-area product. The model sees the name, the JSON schema of the arguments, and the description, and it decides whether the question needs that call.

Earth Observation example, not the geocoder in the lab notebook: a STAC search.

- **Name:** `search_scenes`
- **Arguments:** `collection` (for example `sentinel-2-l2a`), `bbox` as `west,south,east,north`, `start`, `end`, and `max_cloud` 
- **Returns:** a short JSON list: scene id, datetime, cloud cover, and a thumbnail URL. Not the raster itself. The model has to read the result.

This is what you build today. The lab, the registry, and the pull request are all about tools.

### Resources: the client reads them

A resource is data addressed by a URI. File contents, a dataset licence, the GeoJSON of an area the team already agreed on. The host reads it to give the model context. A tool is a call the model chooses. A resource is context the host loads.

Earth Observation example: `resource://server/dem-version` returns a few lines stating that this server uses Copernicus DEM GLO-30, the 2023 release, and a link to the collection. Or `resource://aoi/evia` returns the polygon the briefing is about, so every tool on that server clips to the same geometry.

Use a resource for context that should be available up front. Use a tool for a query whose arguments depend on the question.

### Prompts: a person invokes them

A prompt is a reusable message template, with arguments. A user, or the client's UI, asks for it. It is not a tool the model fires on its own while answering.

Earth Observation example: `wildfire_briefing(place)` returns a user message that asks for the location, what the fire product reports, what recent news says, and which source each sentence came from. The same template can be offered to EVE, Claude, or Cursor. The wording stays stable even when the tools behind it change.

### How a database server would use all three

The architecture overview uses this picture, and it transfers directly. A server about one catalog might expose:

- **Tools** to search scenes and to summarise one item
- **A resource** with the collection's band names and licence
- **A prompt** that asks for a before/after note citing item ids

You do not need all three for the hackathon. A single well-described tool that answers a real question is enough for the pull request. Resources and prompts are there when the context or the wording should travel with the server.


## 3. One small server, three primitives

The next cell builds a server that does not call the network. It is here so you can see the three primitives side by side. [FastMCP](https://gofastmcp.com/servers/tools) reads the function name, the type hints, and the docstring, and turns them into the schema the model sees.

- `@mcp.tool()` registers a function the model can call. The docstring is the description. Arguments without defaults are required.
- `@mcp.resource("uri")` registers data the client can read. The URI is the address. See the [resources guide](https://gofastmcp.com/servers/resources).
- `@mcp.prompt()` registers a template. A returned string becomes a user message. See the [prompts guide](https://gofastmcp.com/servers/prompts).

Run the cell. You should see one tool, one resource, and one prompt, then a sample area for a box around central Evia.


In [ ]:
import json

from fastmcp import FastMCP

demo = FastMCP("Scene helpers")


@demo.tool()
def bbox_area_km2(west: float, south: float, east: float, north: float) -> str:
    """Approximate the area of a longitude/latitude box, in square kilometres.

    Uses 111 km per degree of latitude, and scales longitude by the cosine of
    the mid-latitude. This is a sketch for a tool description, not a geodesy library.

    Args:
        west: Western edge, in degrees.
        south: Southern edge, in degrees.
        east: Eastern edge, in degrees.
        north: Northern edge, in degrees.

    Returns:
        JSON with width_km, height_km, and area_km2.
    """
    import math

    if east <= west or north <= south:
        return json.dumps({"error": "Box is empty. Check that east > west and north > south."})

    mid_lat = (south + north) / 2
    height_km = (north - south) * 111.0
    width_km = (east - west) * 111.0 * math.cos(math.radians(mid_lat))
    return json.dumps(
        {
            "width_km": round(width_km, 1),
            "height_km": round(height_km, 1),
            "area_km2": round(abs(width_km * height_km), 1),
        }
    )


@demo.resource("resource://server/notes")
def scene_notes() -> str:
    """Short note on how this demo server treats coordinates."""
    return (
        "Coordinates are WGS84 degrees. "
        "Bounding boxes are west,south,east,north. "
        "Areas use the 111 km-per-degree approximation."
    )


@demo.prompt()
def briefing_request(place: str, product: str) -> str:
    """Ask for a short Earth Observation note that cites its sources.

    Args:
        place: Place the note is about.
        product: Dataset or service the note should use.
    """
    return (
        f"Write a short note about {place}. "
        f"Use {product} for the measurements. "
        "Include the numbers, the time window, and which tool each fact came from."
    )


print("tools:", [tool.name for tool in await demo.list_tools()])
print("resources:", [str(resource.uri) for resource in await demo.list_resources()])
print("prompts:", [prompt.name for prompt in await demo.list_prompts()])
print(bbox_area_km2(23.0, 38.0, 24.5, 39.0))


## 4. A tool the model can actually use

The decorator does four things, which is why the docstring and the types are part of the tool, not comments for humans:

1. It builds a JSON schema from the signature, so the model knows argument names and types.
2. It uses the docstring as the description the agent reads when choosing a tool. If you pass `description=` to the decorator, that string replaces the docstring for the description. Parameter lines in the docstring still document each argument.
3. It checks the arguments before your function runs.
4. It returns a structured result, including an error the model can read when the call fails.

Keep the tool small. HTTP calls and parsing belong in a helper. The function with `@mcp.tool()` should be a thin wrapper: validate, call the helper, return JSON.

### A description that fails, and one that works

The model never sees your parameter names in isolation. It sees the description.

Weak:

```python
@mcp.tool()
async def get_data(x: str) -> str:
    """Gets data."""
    ...
```

`x` could be a place, a scene id, or a date. The model will guess.

Stronger, still a different service from the geocoder. This one only checks a cloud-cover number and says whether a scene is usable:

```python
@mcp.tool()
def scene_is_clear(cloud_percent: float, max_cloud: float = 20.0) -> str:
    """Decide whether an optical scene is clear enough to use.

    Args:
        cloud_percent: Cloud cover of the scene, from 0 to 100.
        max_cloud: Highest cloud cover you still accept. Default 20.

    Returns:
        JSON with usable (true or false) and the threshold that was applied.
    """
    if not 0 <= cloud_percent <= 100:
        return json.dumps({"error": "cloud_percent must be between 0 and 100."})
    return json.dumps({
        "usable": cloud_percent <= max_cloud,
        "cloud_percent": cloud_percent,
        "max_cloud": max_cloud,
    })
```

Say the units, the range, and what an error looks like. Return JSON text for anything the model must quote later. A Python dict also works with FastMCP. In this lab, matching `geocode_local` and returning a JSON string keeps every tool in one shape.

### One server with two tools, or two servers

Both are normal. Pick the split that matches who owns the service.

**Two tools, one server.** Search and item lookup for the same STAC API. They share the account, the base URL, and the error style. One pull request, one folder in the registry.

**Two servers.** Geocoding is OpenStreetMap. Burnt areas are EFFIS. They have different providers, so they stay separate. The lab notebook does this: `create_agent([geocode_mcp, "effis", "serpapi"])`. Your server is the object. Registry servers are names. The agent calls across them in one conversation.

Do not wrap a service that is already in the registry unless you are extending it. `pretty_print_mcp_servers` in START_HERE lists what is already there.


## 5. How the messages travel

MCP has two layers. The data layer is the JSON-RPC: discovery, tools, resources, prompts. The transport layer is how those bytes move. The same tool works on either transport. From the [architecture overview](https://modelcontextprotocol.io/docs/learn/architecture):

- **Stdio.** The host starts your program and talks over stdin and stdout. No port, no network. This is how a local server runs on a laptop. Logs go to stderr, so they do not corrupt the protocol stream.
- **Streamable HTTP.** The client POSTs JSON to an HTTP endpoint. The server can stream with server-sent events. This is how a remote server serves many clients, and how the registry runs what you merge. Authentication is normal HTTP (bearer tokens, API keys). The spec recommends OAuth when a server needs a user identity.

The registry contract, also in [CONTRIBUTING.md](https://github.com/eve-esa/mcp-tool-registry/blob/main/CONTRIBUTING.md), is the HTTP form:

```python
mcp = FastMCP("your-server", host="0.0.0.0", port=8000, stateless_http=True)
mcp.run(transport="streamable-http")
```

`stateless_http=True` matches the protocol: each request carries what the server needs. Do not store conversation state in the server and expect the next call to remember it. The agent keeps the conversation. Your tool should answer from its arguments.

In the lab notebook you do not write that `run` line for `geocode_mcp`. `create_agent` starts the FastMCP object for you. When you open the pull request, `server.py` does include it. That is the difference between trying a tool in the notebook and publishing it.


## 6. Discovery, and why the registry exists

A client asks the server what it offers, then calls. That is why adding a tool needs no change in EVE, Claude, or Cursor: the next session lists the new tool and the model can choose it.

The [EVE MCP tool registry](https://github.com/eve-esa/mcp-tool-registry) is the catalogue of those servers for Earth Observation. A merged pull request puts your folder under `servers/<name>/`. After that, the lab (and EVE) can refer to it by name, the same way this notebook's companion calls `"effis"` and `"serpapi"`.

A few habits make that hand-off work:

- One server per pull request, so review and CI stay about one service.
- Secrets in the environment, declared as placeholders in `.env.template`. Never commit a key.
- Pin every dependency in `requirements.txt`.
- Descriptions that say what the tool returns and what a failure looks like. The model writes the next call from that text.
- Compact outputs. A bounding box, a table of hectares, a scene id. Not a raw raster in the tool result.

The steps to fork, branch, and open the pull request are in [START_HERE.ipynb](START_HERE.ipynb), section 6.


## 7. The ReAct loop

MCP gives the model tools. It does not say how the model should use them. This lab uses **ReAct** (Reason + Act), from Yao et al., 2023 ([arXiv:2210.03629](https://arxiv.org/abs/2210.03629)).

The model alternates between **reasoning** about what it still needs and **acting** with a tool call. The host executes the call and returns the observation. The model reads that observation and either calls something else or writes the final answer. It stops when it can answer without another tool.

<img src="./images/react-loop.svg" alt="The ReAct loop: the LLM reasons, calls a tool, observes the result, and repeats until a final response is ready" style="max-width: 700px;" />

*Fig. 2. The ReAct loop: the LLM reasons, calls a tool, observes the result, and repeats until a final response is ready.*

`show_trace` in the lab notebook prints this loop: each thought, each tool name, the arguments, and the observation.

### One question, several observations

The worked example in START_HERE asks for a wildfire briefing for Evia and passes three servers: `geocode_mcp`, `"effis"`, and `"serpapi"`. A trace for that question typically looks like this. Wording varies. The shape does not.

1. **Reason.** The fire tool wants a box, and the question only names an island. Geocode first.
2. **Act.** `geocode_local` with `place_name="Evia, Greece"`.
3. **Observe.** JSON whose `top_result.bbox` is `west,south,east,north`.
4. **Reason.** That string is the argument EFFIS expects. Do not invent a box.
5. **Act.** `get_effis_burnt_areas` with that bbox.
6. **Observe.** Hectares, commune, country, fire date, or an empty set.
7. **Reason.** The briefing also asks what recent news says. That is a different server.
8. **Act.** The search tool on `serpapi`.
9. **Observe.** Headlines and links.
10. **Answer.** A short note. Each fact names the tool it came from. If a tool returned nothing, the answer says so.

Nothing in that sequence is hard-coded. The system prompt tells the model to use tools and not to invent data. The tool descriptions tell it which arguments exist. A bad docstring, or a bbox in an unexpected order, shows up here as a wrong call or a second attempt. Fix the tool, run the question again, read the trace.

If the model answers with numbers that appear in no observation, the prompt is too weak or the tool output was too large to use. Shrink the output and say, in the prompt, to cite the tool for every fact.

### What you do, and what the host does

You write the server and the question. The host (EVE, through `create_agent`) owns the loop:

1. `create_agent(servers, prompt=...)` connects the servers, lists their tools, and builds the ReAct agent with your system prompt.
2. `chat(question)` runs the loop and keeps the history, so a follow-up can refer to the previous answer.
3. `show_trace(...)` renders the steps.
4. `close()` stops the servers the notebook started.

Use one agent at a time. Call `close()` before creating another with a fresh history.


## 8. Back to the lab

In [START_HERE.ipynb](START_HERE.ipynb):

- Section 3 defines `geocode_mcp`, the pattern for a tool: type hints, a docstring, JSON out, a clear error.
- Section 4 runs that server together with `effis` and `serpapi` and shows the trace.
- Section 5 is where you implement. Example use cases are in the appendix of that notebook.
- Section 6 is the pull request to the [EVE MCP tool registry](https://github.com/eve-esa/mcp-tool-registry).

Organizers review the notebook on your SageMaker instance. If you edit code on your laptop, upload it back there. The instance stays available for a couple of days after the hackathon so you can download what you produced.
